# Stage 1: Data Lake Foundation — AgriFlow Farming
**ISM 6562 — Final Project | Team: Docker Locker**

## Business Context
AgriFlow Farming is a precision-agriculture company managing thousands of acres across multiple farms.
They face two core challenges: **crop yield prediction** and **irrigation optimization**.
Their raw operational data — sensor telemetry, weather feeds, crop records, and irrigation logs — currently
lives in siloed CSV/JSON exports with no unified storage layer.

This notebook builds AgriFlow's HDFS data lake:
- Designs a three-zone directory structure (landing → curated → analytics)
- Downloads the raw datasets from the course data repo
- Loads them into the landing zone with appropriate file formats
- Sets replication factors based on data criticality
- Verifies the lake structure and documents architecture decisions

---
## 0. Environment Setup

In [ ]:
import subprocess
import urllib.request
import urllib.error
import json
import os

# ── HDFS helper ──────────────────────────────────────────────────────────────
# All hdfs commands run inside the namenode container.
# Returns (stdout, stderr, returncode) as strings.

def hdfs(cmd: str):
    """Run an HDFS command inside the namenode container and print output."""
    full_cmd = f"docker exec agriflow-namenode hdfs {cmd}"
    result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        # HDFS prints INFO to stderr — only show real errors
        errors = [l for l in result.stderr.splitlines()
                  if not any(skip in l for skip in ['INFO', 'WARN', 'SLF4J', 'log4j'])]
        if errors:
            print("STDERR:", '\n'.join(errors))
    return result

# ── Verify HDFS is reachable ─────────────────────────────────────────────────
print("=== HDFS Cluster Status ===")
hdfs("dfsadmin -report | head -20")

---
## 1. Download AgriFlow Raw Data

The course data repository is at:
`https://github.com/prof-tcsmith/6562S26-data/tree/main/final-projects/03-agriflow-farming`

We use the GitHub Contents API to discover all files in the directory,
then download each one to a local staging area before pushing to HDFS.

In [ ]:
REPO_API = (
    "https://api.github.com/repos/prof-tcsmith/6562S26-data"
    "/contents/final-projects/03-agriflow-farming"
)
RAW_BASE = (
    "https://raw.githubusercontent.com/prof-tcsmith/6562S26-data"
    "/main/final-projects/03-agriflow-farming"
)

LOCAL_STAGE = "/home/jovyan/data/agriflow-raw"
os.makedirs(LOCAL_STAGE, exist_ok=True)

# ── Discover files via GitHub API ────────────────────────────────────────────
print("Fetching file list from GitHub API...")
req = urllib.request.Request(REPO_API,
      headers={"Accept": "application/vnd.github+json",
               "User-Agent": "ism6562-agriflow"})
try:
    with urllib.request.urlopen(req) as resp:
        contents = json.loads(resp.read())
    # Filter to files only (skip subdirectories)
    files = [item for item in contents if item["type"] == "file"]
    print(f"Found {len(files)} file(s):")
    for f in files:
        print(f"  {f['name']}  ({f['size']:,} bytes)")
except urllib.error.HTTPError as e:
    print(f"GitHub API error: {e.code} — falling back to known filenames")
    # Fallback: known AgriFlow filenames (update if repo changes)
    files = [
        {"name": "farm_sensors.csv.gz",    "size": 0},
        {"name": "crop_records.csv.gz",    "size": 0},
        {"name": "weather_data.csv.gz",    "size": 0},
        {"name": "irrigation_logs.csv.gz", "size": 0},
        {"name": "soil_samples.csv.gz",    "size": 0},
    ]

In [ ]:
# ── Download each file to local staging ──────────────────────────────────────
downloaded = []

for f in files:
    fname = f["name"]
    local_path = os.path.join(LOCAL_STAGE, fname)
    url = f"{RAW_BASE}/{fname}"

    if os.path.exists(local_path):
        size = os.path.getsize(local_path)
        print(f"  [skip] {fname} already exists ({size:,} bytes)")
        downloaded.append((fname, local_path))
        continue

    print(f"  Downloading {fname} ... ", end="", flush=True)
    try:
        urllib.request.urlretrieve(url, local_path)
        size = os.path.getsize(local_path)
        print(f"{size:,} bytes")
        downloaded.append((fname, local_path))
    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nDownloaded {len(downloaded)} file(s) to {LOCAL_STAGE}")

---
## 2. Peek at the Data

Before designing the HDFS zone structure we need to understand what we have.
Spark reads `.gz` files natively, so we spin up a minimal SparkSession just
to inspect schemas — no HDFS needed yet.

In [ ]:
from pyspark.sql import SparkSession

# ── SparkSession with HDFS connectivity ──────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("AgriFlow-Stage1-DataLake")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    # Tell Spark where HDFS lives so hdfs:// paths resolve correctly
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .getOrCreate()
)

print(f"Spark version : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")
print(f"App ID        : {spark.sparkContext.applicationId}")

In [ ]:
# ── Inspect each downloaded file ─────────────────────────────────────────────
raw_dfs = {}   # {filename: DataFrame} — kept for Stage 2

for fname, local_path in downloaded:
    print(f"\n{'='*60}")
    print(f"FILE: {fname}")
    print(f"{'='*60}")

    try:
        if fname.endswith(".json.gz") or fname.endswith(".json"):
            df = spark.read.option("multiline", "false").json(local_path)
        else:  # csv or csv.gz
            df = (
                spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(local_path)
            )

        print(f"  Rows   : {df.count():,}")
        print(f"  Columns: {len(df.columns)}")
        df.printSchema()
        df.show(3, truncate=True)

        key = fname.replace(".csv.gz", "").replace(".json.gz", "") \
                   .replace(".csv", "").replace(".json", "")
        raw_dfs[key] = df

    except Exception as e:
        print(f"  ERROR reading {fname}: {e}")

---
## 3. HDFS Zone Design

### Architecture Decision

We adopt a **three-zone data lake** model:

| Zone | HDFS Path | Contents | Format | Replication |
|------|-----------|----------|--------|-------------|
| **Landing** | `/agriflow/landing/` | Raw files as-is from source systems | `.csv.gz` / `.json.gz` | 2 |
| **Curated** | `/agriflow/curated/` | Cleaned, typed, validated data (Stage 2 output) | Parquet | 2 |
| **Analytics** | `/agriflow/analytics/` | Aggregated answers to business questions (Stage 2 output) | Parquet (partitioned) | 1 |

### File Format Justification

**Landing zone — keep original `.csv.gz` / `.json.gz`:**
- Spark reads gzip natively with zero extra tooling
- Preserves the exact source format for auditability and reprocessing
- gzip compresses CSV ~5–10×, keeping ingest storage low
- Tradeoff: gzip is non-splittable → single-task read; acceptable at AgriFlow's
  150–200 MB source size; would switch to `.csv.lz4` at 10+ GB per file

**Curated / Analytics zones — Parquet:**
- Columnar storage means Spark's optimizer prunes columns before reading
- Schema embedded in footer → no schema-on-read ambiguity
- Snappy compression (Parquet default) is splittable and fast to decompress
- Week 9 assignment showed 10.75× compression vs raw CSV — relevant here

### Replication Factor Justification

| Data | RF | Reasoning |
|------|----|-----------|
| Landing (sensor, weather, crop) | 2 | Operational data; loss is recoverable by re-pulling from source; RF=2 doubles fault tolerance vs RF=1 within our 2-datanode cluster |
| Curated (cleaned) | 2 | Expensive Spark transformations make re-computation costly; keep 2 copies |
| Analytics (aggregates) | 1 | Derived outputs easily regenerated by re-running the DAG; saving space for raw data |

*In production (3+ nodes), sensor telemetry and soil samples would use RF=3 given irrigation decisions depend on them.*

---
## 4. Create HDFS Directory Structure

In [ ]:
# ── HDFS zone directories ─────────────────────────────────────────────────────
# Mirror the logical data sources from the AgriFlow scenario

HDFS_DIRS = [
    # Landing zone — one subdirectory per data source
    "/agriflow/landing/farm-sensors",
    "/agriflow/landing/crop-records",
    "/agriflow/landing/weather-data",
    "/agriflow/landing/irrigation-logs",
    "/agriflow/landing/soil-samples",
    # Curated zone — Stage 2 will write here
    "/agriflow/curated/sensors-clean",
    "/agriflow/curated/crops-clean",
    "/agriflow/curated/weather-clean",
    "/agriflow/curated/irrigation-clean",
    "/agriflow/curated/soil-clean",
    # Analytics zone — Stage 2 aggregated outputs
    "/agriflow/analytics/yield-by-crop",
    "/agriflow/analytics/irrigation-efficiency",
    "/agriflow/analytics/sensor-anomalies",
    "/agriflow/analytics/weather-correlation",
]

print("Creating HDFS directories...")
for d in HDFS_DIRS:
    r = hdfs(f"dfs -mkdir -p {d}")
    if r.returncode == 0:
        print(f"  [OK] {d}")
    else:
        print(f"  [ERR] {d}")

In [ ]:
# ── Verify the tree was created ───────────────────────────────────────────────
print("=== HDFS /agriflow directory tree ===")
hdfs("dfs -ls -R /agriflow")

---
## 5. Load Raw Data into Landing Zone

In [ ]:
# ── Map filenames to their landing zone subdirectory ─────────────────────────
# Update this mapping if the course repo uses different filenames.

LANDING_MAP = {
    "farm_sensors":    "/agriflow/landing/farm-sensors",
    "crop_records":    "/agriflow/landing/crop-records",
    "weather_data":    "/agriflow/landing/weather-data",
    "irrigation_logs": "/agriflow/landing/irrigation-logs",
    "soil_samples":    "/agriflow/landing/soil-samples",
}

# ── Copy local staging files → namenode → HDFS ───────────────────────────────
# We first copy the file into the namenode container, then use hdfs -put.
# This avoids needing HDFS client libraries on the Jupyter host.

print("Loading files into HDFS landing zone...\n")

for fname, local_path in downloaded:
    # Determine the base key (strip extensions)
    key = fname.replace(".csv.gz", "").replace(".json.gz", "") \
               .replace(".csv", "").replace(".json", "")

    # Match key to landing directory (partial match for flexibility)
    hdfs_dir = None
    for pattern, path in LANDING_MAP.items():
        if pattern in key:
            hdfs_dir = path
            break

    if hdfs_dir is None:
        # Default to a generic landing area if no pattern matches
        hdfs_dir = "/agriflow/landing/misc"
        hdfs(f"dfs -mkdir -p {hdfs_dir}")

    hdfs_path = f"{hdfs_dir}/{fname}"

    # Check if already in HDFS (idempotent)
    check = hdfs(f"dfs -test -e {hdfs_path}")
    if check.returncode == 0:
        print(f"  [skip] {fname} already in HDFS")
        continue

    # Step 1: copy local file into namenode container
    cp_cmd = (
        f"docker cp {local_path} "
        f"agriflow-namenode:/tmp/{fname}"
    )
    r = subprocess.run(cp_cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  [ERR] docker cp failed for {fname}: {r.stderr}")
        continue

    # Step 2: put from namenode local filesystem → HDFS
    r = hdfs(f"dfs -put /tmp/{fname} {hdfs_path}")
    if r.returncode == 0:
        print(f"  [OK] {fname} → {hdfs_path}")
    else:
        print(f"  [ERR] {fname} failed to upload")

print("\nDone.")

---
## 6. Set Replication Factors

In [ ]:
# ── Apply zone-level replication policies ────────────────────────────────────
# See Section 3 (Zone Design) for rationale.

print("Setting replication factors...\n")

# Landing zone — RF=2 (sensor + crop data is operationally critical)
hdfs("dfs -setrep -w 2 /agriflow/landing")
print("  Landing zone: RF=2")

# Curated zone — RF=2 (expensive transforms, worth protecting)
hdfs("dfs -setrep -w 2 /agriflow/curated")
print("  Curated zone: RF=2")

# Analytics zone — RF=1 (easily regenerated; save space)
hdfs("dfs -setrep -w 1 /agriflow/analytics")
print("  Analytics zone: RF=1")

---
## 7. Verify & Inventory

In [ ]:
# ── Landing zone inventory with file sizes ───────────────────────────────────
print("=== Landing Zone Inventory ===")
hdfs("dfs -du -h /agriflow/landing")

In [ ]:
# ── Confirm replication factors on actual blocks ─────────────────────────────
print("=== Replication Status (landing) ===")
hdfs("fsck /agriflow/landing -files -blocks | grep -E '(Status|repl|HEALTHY|missing)' | head -30")

In [ ]:
# ── Read a landing-zone file back through Spark to confirm round-trip ────────
print("=== Round-trip Read Test (HDFS → Spark) ===")

# Find the first CSV file in the landing zone
ls = hdfs("dfs -ls -R /agriflow/landing")
hdfs_files = [line.split()[-1] for line in ls.stdout.splitlines()
              if line.strip() and not line.startswith('d')
              and 'agriflow/landing' in line]

if hdfs_files:
    test_file = hdfs_files[0]
    print(f"\nReading: hdfs://namenode:9000{test_file}")

    if test_file.endswith(".json.gz") or test_file.endswith(".json"):
        df_test = spark.read.json(f"hdfs://namenode:9000{test_file}")
    else:
        df_test = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(f"hdfs://namenode:9000{test_file}")
        )

    print(f"Rows: {df_test.count():,}  |  Columns: {len(df_test.columns)}")
    df_test.printSchema()
    df_test.show(5, truncate=True)
else:
    print("No files found in landing zone — check upload step above.")

---
## 8. Architecture Summary

| Item | Decision | Justification |
|------|----------|---------------|
| Zone model | 3-zone (landing / curated / analytics) | Separates raw ingestion from transformation and reporting; enables reprocessing without data loss |
| Landing format | `.csv.gz` / `.json.gz` (as-is from source) | Zero-copy ingest; Spark reads natively; preserves provenance |
| Curated format | Parquet + Snappy | Columnar = fast analytical queries; schema embedded; splittable |
| Analytics format | Parquet (partitioned by crop type or date) | Partition pruning reduces scan size for dashboard queries |
| Landing RF | 2 | Sensor and crop data drives irrigation decisions — loss is operationally costly |
| Curated RF | 2 | Re-running Spark transforms is expensive; replication is cheap insurance |
| Analytics RF | 1 | Easily regenerated from curated zone; conserves block storage |
| DataNodes | 2 | Matches `dfs.replication=2`; minimum for any redundancy |

**Next: Stage 2** will read from `/agriflow/landing/`, clean and join the datasets
using PySpark, and write curated Parquet to `/agriflow/curated/`.

In [ ]:
spark.stop()
print("SparkSession stopped. Stage 1 complete.")